# Executor Failures: Node Eviction, Heartbeats, and State Handling

**Module**: 11_fault_tolerance_and_reliability  
**Notebook**: 02_executor_failure.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: What happens when an executor process crashes: heartbeat timeout, cluster manager notification, task rescheduling, and shuffle data loss.
- **Why does this topic exist?**: Distributed systems must be designed for frequent hardware and container failures.
- **What problem does it solve?**: Understanding why shuffle files may force upstream stage recomputation.
- **Where is it used in real-world Data Engineering?**: Running Spark on AWS EC2 Spot Instances.

---

## 2. Executor Failure Lifecycle
1. Heartbeat stops (`spark.network.timeout` = 120s).
2. Master marks executor dead; requests replacement.
3. Running tasks rescheduled on healthy workers.
4. Shuffle files lost -> Upstream ShuffleMapStage recomputed.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Executor_Failure_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
conf = spark.sparkContext.getConf()
print("spark.network.timeout:         ", conf.get("spark.network.timeout", "120s"))
print("spark.executor.heartbeatInterval: ", conf.get("spark.executor.heartbeatInterval", "10s"))

---

## Experiment: External Shuffle Service Resilience

- **1. What are we testing?**: Explain role of External Shuffle Service during executor loss.
- **2. What do we expect?**: ESS preserves shuffle files after executor crash.

In [ ]:
print('ESS runs as an auxiliary node daemon so shuffle files survive executor JVM crashes.')

**3. What happened?**: Explained ESS architecture.  
**4. Why did it happen?**: ESS decouples shuffle storage from executor process lifecycle.  
**5. Practical Engineering Lesson**: Enable ESS when running dynamic allocation or preemptible instances.

---

## Practical Exercises

### Exercise 1 (Beginner): Check Heartbeat Interval
**Problem Statement**: Print default heartbeat interval.  
**Input / Context**: conf

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Stage Invalidation
**Problem Statement**: Why lost shuffle files invalidate stages.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Dynamic Allocation with ESS
**Problem Statement**: Why dynamic allocation requires ESS.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Check Heartbeat Interval):
```python
print(spark.conf.get('spark.executor.heartbeatInterval','10s'))
```

#### Solution 2 (Stage Invalidation):
```python
print('Reducers cannot fetch missing shuffle map files; parent stage must re-run.')
```

#### Solution 3 (Dynamic Allocation with ESS):
```python
print('Without ESS, deallocated executors would delete their shuffle files.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Executor Crash**: Driver automatically reschedules tasks and recomputes lost data.
- **Shuffle Loss**: Forces upstream stage rerun unless ESS is used.

### Essential Technical Terms
- `Heartbeat`: Periodic health ping from executor to driver.
- `ESS`: External Shuffle Service.

### Common Engineering Mistakes
- **Assuming application dies with executor**: Spark applications survive executor crashes gracefully.

### Interview Questions on This Topic
- **Q: What happens if an executor holding shuffle files crashes?**
  - *A*: The DAGScheduler invalidates the upstream ShuffleMapStage and reschedules the lost map tasks to regenerate the shuffle files.

### What to Study Next
Open 03_task_retries.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')